### Transformações das tabelas Dimensões (Camada Silver)
**🔥 O que esse código faz?**
- ✅ Lê todos arquivos delta dentro bronze/dimensao .
- ✅ Transforma cada arquivo em Delta Parquet e salva na pasta correspondente (bronze/).
- ✅ Realiza as transformações conforme escopo
> -     Tratar a colunas relativas aos nomes de cliente, unificando em 1 única coluna “Nome”
> -     Tratar a colunas relativas aos nomes de vendedor, unificando em 1 única coluna “Nome”
> -     Eliminar a coluna NumeroTransacao de vendas
> -     Remoção de colunas não utilizadas (relativa a nomes) 
- ✅ Salva arquivos delta parquet na camada silver/dimensao
- ✅ Mantém a flexibilidade para qualquer tabela sem precisar mudar o código.


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Configuração inicial da SparkSession com configurações otimizadas
spark = SparkSession.builder \
    .appName("Load Data Silver") \
    .config("spark.sql.shuffle.partitions", "200")  \
    .config("spark.sql.files.maxPartitionBytes", "1GB") \
    .config("spark.sql.files.maxRecordsPerFile", "1000000") \
    .config("spark.sql.parquet.compression.codec", "snappy") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

#### Configuração de Parametros e Caminhos

In [0]:
# Define o caminho base do Data Lake
base_path = '/mnt/panex/lhdw'

# Define os caminhos das camadas Bronze e Silver
bronze_path = f"{base_path}/bronze/"
silver_path = f"{base_path}/silver"
gold_path_fato = f"{base_path}/gold/fato"
tipo_dim ="dimensao"
tipo_fato ="fato"

### Transformação dos Dados: Validação de qualidade dos dados
- ✅ **Verificar nulos**: Identificar registros com valores NULL nas colunas obrigatórias.
- ✅ **Verificar dados inválidos**: Pode incluir regras específicas, como verificar se CategoriaID é negativo ou se NomeCategoria está vazio.
- ✅ **Adicionar coluna** data_carga: Inserir um timestamp com a data e hora atuais.

### Transformação - Dimensão Categoria

In [0]:
# Parametro de tabela 
tabela = "categorias"
# Leitura de arquivo Delta na camada bronze
df = spark.read.format("delta").load(f"{bronze_path}/{tipo_dim}/{tabela}")

# Transformações e validações
# Adiciona uma coluna com a data e hora atual
df = df.withColumn("data_carga", F.current_timestamp())

# Preenche valores nulos
df = df.fillna({
    "CategoriaID": -1,          # Define -1 para valores nulos
    "NomeCategoria": "N/A"      # Preenche nome vazio com "N/A"
})

# Persistir arquivo Delta parquet na camada Silver
df.write.format("delta")\
    .mode("overwrite") \
    .save(f"{silver_path}/{tipo_dim}/{tabela}")

# Liberando a memória após salvar os DataFrames
df.unpersist()

Out[3]: DataFrame[CategoriaID: int, NomeCategoria: string, data_carga: timestamp]

### Transformação - Dimensão Produtos

In [0]:
# Parametro de tabela 
tabela = "produtos"
# Leitura de arquivo Delta na camada bronze
df = spark.read.format("delta").load(f"{bronze_path}/{tipo_dim}/{tabela}")

# Transformações e validações
# Adiciona uma coluna com a data e hora atual
df = df.withColumn("data_carga", F.current_timestamp())

# Preenche valores nulos
df = df.fillna({
    "ProdutoID": -1,          # Define -1 para valores nulos
    "ProdutoNome": "N/A",     # Preenche ProdutoNome vazio com "N/A"
    "Preco": 0,               # Preenche preço vazio com 0
    "CategoriaID": -1,        # Preenche CategoriaID vazio com -1
})

# Persistir arquivo Delta parquet na camada Silver
df.write.format("delta")\
    .mode("overwrite") \
    .save(f"{silver_path}/{tipo_dim}/{tabela}")

# Liberando a memória após salvar os DataFrames
df.unpersist()

Out[4]: DataFrame[ProdutoID: int, ProdutoNome: string, Preco: double, CategoriaID: int, Classe: string, DataCadastro: timestamp, Resistencia: string, EAlergico: string, ValidadeDias: double, data_carga: timestamp]

### Transformação - Dimensão Países

In [0]:
# Parametro de tabela 
tabela = "paises"
# Leitura de arquivo Delta na camada bronze
df = spark.read.format("delta").load(f"{bronze_path}/{tipo_dim}/{tabela}")
#Transformações e validações
# Adiciona uma coluna com a data e hora atual
df = df.withColumn("data_carga", F.current_timestamp())

# Preenche valores nulos
df = df.fillna({
    "PaisID": -1,       # Define -1 para valores nulos
    "PaisNome": "N/A",  # Preenche PaisNome vazio com "N/A"
    "SiglaPais":"N/A"   # Preenche SiglaPais vazio com "N/A"
   })

# Persistir arquivo Delta parquet na camada Silver
df.write.format("delta")\
    .mode("overwrite") \
    .save(f"{silver_path}/{tipo_dim}/{tabela}")

# Liberando a memória após salvar os DataFrames
df.unpersist()

Out[5]: DataFrame[PaisID: int, PaisNome: string, SiglaPais: string, data_carga: timestamp]

### Transformação - Dimensão Cidades

In [0]:
# Parametro de tabela 
tabela = "cidades"
# Leitura de arquivo Delta na camada bronze
df = spark.read.format("delta").load(f"{bronze_path}/{tipo_dim}/{tabela}")
#Transformações e validações
# Adiciona uma coluna com a data e hora atual
df = df.withColumn("data_carga", F.current_timestamp())

# Preenche valores nulos
df = df.fillna({
    "CidadeID": -1,       # Define CidadeID -1 para valores nulos
    "NomeCidade": "N/A",  # Preenche NomeCidade vazio com "N/A"
    "PaisID":-1           # Preenche PaisID -1 para valores nulos
   })

# Persistir arquivo Delta parquet na camada Silver
df.write.format("delta")\
    .mode("overwrite") \
    .save(f"{silver_path}/{tipo_dim}/{tabela}")

# Liberando a memória após salvar os DataFrames
df.unpersist()

Out[6]: DataFrame[CidadeID: int, NomeCidade: string, Cep: int, PaisID: int, data_carga: timestamp]

### Transformação - Dimensão Clientes

In [0]:
# Parametro de tabela 
tabela = "clientes"
# Leitura de arquivo Delta na camada bronze
df = spark.read.format("delta").load(f"{bronze_path}/{tipo_dim}/{tabela}")
#Transformações e validações
# Adiciona uma coluna com a data e hora atual
df = df.withColumn("data_carga", F.current_timestamp())

# Preenche valores nulos
df = df.fillna({
    "ClienteID": -1,        # Define ClienteID-1 para valores nulos
    "PrimeiroNome": "N/A",  # Preenche PrimeiroNome vazio com "N/A"
    "CidadeID":-1           # Preenche CidadeID -1 para valores nulos
   })

# Unifica as colunas de nome
df = df.withColumn("Nome", 
    F.concat_ws(" ", F.col("PrimeiroNome"), F.col("NomeDoMeio"), F.col("UltimoNome"))
)

# Remove espaços extras gerados por colunas vazias
df = df.withColumn("Nome", F.trim(F.col("Nome")))

# Remove as colunas antigas
df = df.drop("PrimeiroNome", "NomeDoMeio", "UltimoNome")
# Persistir arquivo Delta parquet na camada Silver
df.write.format("delta")\
    .mode("overwrite") \
    .save(f"{silver_path}/{tipo_dim}/{tabela}")

# Liberando a memória após salvar os DataFrames
df.unpersist()

Out[7]: DataFrame[ClienteID: int, CidadeID: int, Endereco: string, data_carga: timestamp, Nome: string]

### Transformação - Dimensão Vendedores

In [0]:
# Parametro de tabela 
tabela = "vendedores"
# Leitura de arquivo Delta na camada bronze
df = spark.read.format("delta").load(f"{bronze_path}/{tipo_dim}/{tabela}")
#Transformações e validações
# Adiciona uma coluna com a data e hora atual
df = df.withColumn("data_carga", F.current_timestamp())

# Preenche valores nulos
df = df.fillna({
    "VendedorID": -1,        # Define ClienteID-1 para valores nulos
    "PrimeiroNome": "N/A",   # Preenche PrimeiroNome vazio com "N/A"
    "CidadeID":-1            # Preenche CidadeID -1 para valores nulos
   })

# Unifica as colunas de nome
df = df.withColumn("Nome", 
    F.concat_ws(" ", F.col("PrimeiroNome"), F.col("NomeDoMeio"), F.col("UltimoNome"))
)

# Remove espaços extras gerados por colunas vazias
df = df.withColumn("Nome", F.trim(F.col("Nome")))

# Remove as colunas antigas
df = df.drop("PrimeiroNome", "NomeDoMeio", "UltimoNome")

# Persistir arquivo Delta parquet na camada Silver
df.write.format("delta")\
    .mode("overwrite") \
    .save(f"{silver_path}/{tipo_dim}/{tabela}")

# Liberando a memória após salvar os DataFrames
df.unpersist()

Out[8]: DataFrame[VendedorID: int, DataNascimento: timestamp, Genero: string, CidadeID: int, DataAdmissao: timestamp, data_carga: timestamp, Nome: string]

%md
### Transformação -  Fato ( Camada Silver)
**🔥 O que esse etapa faz?**
- ✅ Lê todos arquivos delta dentro bronze/fato .
- ✅ Remoção da coluna NumeroTransacao.
- ✅ Trazer o valor unitario da tabela produto.
- ✅ Cálculo do PrecoTotal multiplicando a (quantidade pelo preço unitário) do produto na data da venda e aplicando o desconto.
- ✅ Cria as colunas Ano e Mes a partir da DataVenda para particionamento.
- ✅ Implementa carga incremental com base no ultimo VendasID
- ✅ Para Fato Carga: aquivos Delta Parquet particionado em Ano/Mes

In [0]:
# Parâmetro de tabela dimensão
tabela = "produtos"
# Leitura de arquivo Delta na camada bronze
df_prod = spark.read.format("delta").load(f"{bronze_path}/{tipo_dim}/{tabela}")

# Verifica se a camada Gold existe (usando dbutils)
try:
    dbutils.fs.ls(gold_path)  # Se o diretório existe, não gera erro
    gold_exists = True
except:
    gold_exists = False  # Se o diretório não existir, definimos como False

# Obtém o maior VendasID já carregado, se a camada Gold existir
if gold_exists:
    df_gold = spark.read.format("delta").load(gold_path)
    max_vendas_id = df_gold.agg(F.max("VendasID")).collect()[0][0] or 0
else:
    max_vendas_id = 0  # Se não existe, considera 0

print(f"🔍 Último VendasID na Gold: {max_vendas_id}")

# Parâmetro de tabela fato - Leitura da camada Bronze
df_vendas = spark.read.format("delta").load(f"{bronze_path}/{tipo_fato}")

# Filtra apenas novos registros para carga incremental
df_vendas_novos = df_vendas.filter(F.col("VendasID") > max_vendas_id)

if df_vendas_novos.count() > 0:
    # Fazendo o JOIN com df_prod (pegando apenas ProdutoID e Preco) e aplicando todas as transformações
    df_vendas_enriquecido = (
        df_vendas_novos
        .join(F.broadcast(df_prod.select("ProdutoID", F.col("Preco").alias("PrecoUnitario"))), "ProdutoID", "left")
        .withColumn("PrecoTotal", F.round((F.col("Quantidade") * F.col("PrecoUnitario")) - F.col("Desconto"), 2))
        .withColumn("DataVenda", F.to_date("DataVenda"))
        .withColumn("Ano", F.year("DataVenda"))  # Adiciona coluna Ano
        .withColumn("Mes", F.month("DataVenda"))  # Adiciona coluna Mes
        .withColumn("data_carga", F.current_timestamp()) #Data Hora Carga
        .drop("NumeroTransacao")  # Remove colunas desnecessárias
    )

    # Grava no modo append para manter os dados históricos
    df_vendas_enriquecido.write.format("delta")\
        .option("mergeSchema", "true")\
        .mode("append")\
        .partitionBy("Ano", "Mes")\
        .save(f"{silver_path}/{tipo_fato}")
    
    print(f"✅ {df_vendas_novos.count()} novos registros adicionados à Silver.")
else:
    print("⚠ Nenhum novo registro para adicionar.")

🔍 Último VendasID na Gold: 0
✅ 965446 novos registros adicionados à Silver.


### 🗑️Limpeza de Memória

In [0]:
# Liberar cache dos DataFrames do Spark
for df in [df_prod, df_vendas, df_vendas_novos, df_vendas_enriquecido]:
    if "df" in locals() and df is not None:
        df.unpersist()  # Remove do cache do Spark
        del df  # Remove referência do Python

# Força a liberação de memória
import gc
gc.collect()

print("✅ Memória liberada e DataFrames descartados.")


✅ Memória liberada e DataFrames descartados.
